In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from src.ml_config import (
    INSURANCE_DATASET_PATH,
    TARGET_FIELDS,
    PREDICTION_FIELD,
    FIELDS_TO_DELETE,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

OUT_DIR = Path("data/output/eda")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def main():
    if not INSURANCE_DATASET_PATH.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {INSURANCE_DATASET_PATH}")

    df = pd.read_csv(INSURANCE_DATASET_PATH)

    section("1. Allgemeine Informationen zum Datensatz")
    print(f"Zeilen: {len(df)}, Spalten: {df.shape[1]}")
    print(df.dtypes)

    # ------------------------------------------------------------------
    section("2. Zielvariable: vehicle_claim — Basisstatistik")
    if PREDICTION_FIELD not in df.columns:
        raise KeyError(f"Spalte {PREDICTION_FIELD} ist im Datensatz nicht vorhanden")

    y = df[PREDICTION_FIELD]
    print(y.describe())
    print(f"\nFehlende Werte in {PREDICTION_FIELD}: {y.isna().sum()}")
    print(f"Nullwerte: {(y == 0).sum()} ({(y == 0).mean():.1%})")
    print(f"Negative Werte: {(y < 0).sum()}")

    # IQR-Methode als Orientierung (nicht zum automatischen Entfernen!)
    q1, q3 = y.quantile(0.25), y.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((y < low) | (y > high)).sum()
    print(f"\nIQR-Grenzen: [{low:,.0f}, {high:,.0f}]")
    print(f"Punkte außerhalb von IQR*1.5: {n_outliers} ({n_outliers/len(y):.1%})")

    # Perzentile, um die Verteilungsenden (Tails) zu sehen
    for p in [0.01, 0.05, 0.5, 0.95, 0.99]:
        print(f"  Perzentil {p:>4.0%}: {y.quantile(p):,.0f}")

    # ------------------------------------------------------------------
    section("3. vehicle_claim nach incident_severity")
    if "incident_severity" in df.columns:
        print(df.groupby("incident_severity")[PREDICTION_FIELD]
                .agg(["count", "mean", "median", "std", "min", "max"])
                .sort_values("mean"))
        print("\n-> Wenn bei 'Trivial Damage' (oder einer ähnlichen Kategorie) die")
        print("   meisten Werte = 0 sind, handelt es sich nicht um Ausreißer, sondern")
        print("   um einen eigenen Modus: Das Modell sollte dies als legitimen")
        print("   'Nullfall' erkennen und nicht als zu entfernendes Rauschen.")

    section("4. vehicle_claim nach collision_type")
    if "collision_type" in df.columns:
        print(df.groupby("collision_type", dropna=False)[PREDICTION_FIELD]
                .agg(["count", "mean", "median", "std"])
                .sort_values("mean"))

    section("5. vehicle_claim nach incident_type")
    if "incident_type" in df.columns:
        print(df.groupby("incident_type", dropna=False)[PREDICTION_FIELD]
                .agg(["count", "mean", "median", "std"])
                .sort_values("mean"))

    # ------------------------------------------------------------------
    section("6. Fehlende Werte in allen Spalten von TARGET_FIELDS + PREDICTION_FIELD")
    cols = TARGET_FIELDS + [PREDICTION_FIELD]
    cols = [c for c in cols if c in df.columns]
    miss = df[cols].isna().sum()
    miss = miss[miss > 0]
    if len(miss):
        print(miss)
    else:
        print("Keine fehlenden Werte (prüfen Sie aber auch Platzhalter wie '?' / 'NA' /")
        print("'UNKNOWN' — in dieser Art von Datensatz (insurance fraud) sind fehlende")
        print("Werte häufig als '?' kodiert.")

    # explizite Prüfung auf '?'-Platzhalter, typisch für diese Datensatzfamilie
    section("6b. Prüfung auf Platzhalter für fehlende Werte ('?')")
    for c in cols:
        if df[c].dtype == object:
            n_q = (df[c] == "?").sum()
            if n_q:
                print(f"  {c}: {n_q} Werte mit '?'")

    # ------------------------------------------------------------------
    section("7. Korrelation der numerischen Merkmale mit der Zielvariable")
    numeric_candidates = [c for c in TARGET_FIELDS
                           if pd.api.types.is_numeric_dtype(df[c])] if all(c in df.columns for c in TARGET_FIELDS) else []
    numeric_candidates = [c for c in numeric_candidates if c in df.columns]
    if numeric_candidates:
        corr = df[numeric_candidates + [PREDICTION_FIELD]].corr(numeric_only=True)[PREDICTION_FIELD]
        print(corr.sort_values(ascending=False))
        print("\n-> Wenn alle Korrelationen nahe 0 liegen, bestätigt das: Das aktuelle")
        print("   Merkmalsset erklärt vehicle_claim nur schwach, und das niedrige R2")
        print("   ist eher eine Folge der Merkmalsauswahl als nur von Ausreißern.")

    # ------------------------------------------------------------------
    section("8. Entfernte Felder (FIELDS_TO_DELETE) — mögliches Leakage vs. nützliches Signal")
    leak_candidates = ["injury_claim", "total_claim_amount", "property_claim"]
    present = [c for c in leak_candidates if c in df.columns]
    if present:
        corr2 = df[present + [PREDICTION_FIELD]].corr(numeric_only=True)[PREDICTION_FIELD]
        print(corr2)
        print("\n-> total_claim_amount korreliert vermutlich stark mit vehicle_claim,")
        print("   da häufig total = injury + property + vehicle gilt. Das ist direktes")
        print("   Leakage — es ist richtig, dass Sie es aus den Merkmalen entfernt haben.")
        print("   injury_claim und property_claim sind separate Komponenten ohne klaren")
        print("   arithmetischen Bezug zu vehicle_claim. Sie könnten daher als Merkmale")
        print("   BEIBEHALTEN werden (kein Leakage), sofern sie zum Vorhersagezeitpunkt")
        print("   bereits bekannt sind.")

    # ------------------------------------------------------------------
    section("9. Diagramme zur visuellen Prüfung speichern")
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(y, bins=50)
        axes[0].set_title("Verteilung von vehicle_claim")
        axes[0].set_xlabel("vehicle_claim")

        axes[1].boxplot(y.dropna(), vert=False)
        axes[1].set_title("Boxplot vehicle_claim")

        fig.tight_layout()
        out_path = OUT_DIR / "vehicle_claim_distribution.png"
        fig.savefig(out_path, dpi=120)
        print(f"Gespeichert: {out_path}")

        if "incident_severity" in df.columns:
            fig2, ax2 = plt.subplots(figsize=(8, 5))
            df.boxplot(column=PREDICTION_FIELD, by="incident_severity", ax=ax2, rot=30)
            ax2.set_title("vehicle_claim nach incident_severity")
            plt.suptitle("")
            fig2.tight_layout()
            out_path2 = OUT_DIR / "vehicle_claim_by_severity.png"
            fig2.savefig(out_path2, dpi=120)
            print(f"Gespeichert: {out_path2}")

    except ImportError:
        print("matplotlib ist nicht installiert — Diagramme werden übersprungen (pip install matplotlib)")

    section("Fertig")
    print("Nächster Schritt: Schauen Sie sich data/output/eda/*.png sowie die")
    print("Abschnitte 2-3-7 oben an, bevor Sie über das Entfernen von Ausreißern entscheiden.")